# Correlation of blood pressure burden on outcome (mRs)

## Preprocessing

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from statsmodels.miscmodels.ordinal_model import OrderedModel

from utils.utils import load_encrypted_xlsx
from bp.bp_burden.analysis_utils import count_events, define_events_multiple_thresholds, event_count_to_mrs_correlation, event_product_to_mrs_correlation, multiple_duration_thresholds, decision_boundary_analysis, total_event_duration, event_relative_duration_to_mrs_correlation, relative_duration_in_correlated_events

In [ ]:
registry_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/sos_sah_data/post_hoc_modified_aSAH_DATA_2009_2023_24122023.xlsx'
bp_df_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/pdms_data/extracted_data/20240116_SAH_SOS_Blutdruecke.csv'
nor_annotated_bd_df_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/pdms_data/extracted_data/20240116_SAH_SOS_Blutdruecke_nor_annotated.csv'
pdms_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/pdms_data/registry_pdms_correspondence.csv'
outcome_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/sos_sah_data/follow_up/aSAH_DATA_2009_2024_18122024.xlsx'

In [ ]:
filter_noradenaline = False

# # T0_reference can be one of 'ictus', 'first_measure'
# T0_reference = 'first_measure'

# if T0_reference not in ['ictus', 'first_measure']:
#     raise ValueError("T0_reference must be either 'ictus' or 'first_measure'.")

In [ ]:
registry_df = load_encrypted_xlsx(registry_data_path)
outcome_df = load_encrypted_xlsx(outcome_path)

if not filter_noradenaline:
    bp_df = pd.read_csv(bp_df_path, sep= ';', decimal='.')
else:
    nor_annotated_bp_df = pd.read_csv(nor_annotated_bd_df_path)
    bp_df = nor_annotated_bp_df[nor_annotated_bp_df['noradrenaline_concomitant'] == 0]

registry_pdms_correspondance_df = pd.read_csv(pdms_path)

In [ ]:
# drop duplicates 
bp_df = bp_df.drop_duplicates(subset=['pNr', 
                                       'systole',
                                       'diastole',
                                       'mitteldruck',
                                       'timeBd'])

registry_df.drop_duplicates(inplace=True)
registry_df.dropna(subset=['SOS-CENTER-YEAR-NO.', 'Name', 'Date_birth'], inplace=True)

In [ ]:
bp_df=bp_df.merge(registry_pdms_correspondance_df, how='left', on='pNr')

bp_df['Date_birth']=pd.to_datetime(bp_df['Date_birth'], format='%d.%m.%Y')
outcome_df['Date_birth']=pd.to_datetime(outcome_df['Date_birth'])

outcome_df["mRS_FU_1y"]=pd.to_numeric(outcome_df['mRS_FU_1y'], errors='coerce')

In [ ]:
for pnr in tqdm(bp_df["pNr"].unique()):
    sos_center_nr = bp_df[bp_df["pNr"] == pnr]["SOS-CENTER-YEAR-NO."].values[0]
    name = bp_df[bp_df["pNr"] == pnr]["JoinedName"].values[0]
    date_birth = bp_df[bp_df["pNr"] == pnr]["Date_birth"].values[0]
    mrs_values = outcome_df[(outcome_df["SOS-CENTER-YEAR-NO."] == sos_center_nr) &
                        (outcome_df["Name"] == name) &
                        (outcome_df["Date_birth"] == date_birth)]["mRS_FU_1y"]
    if len(mrs_values) == 0:
        mrs = np.nan
    else:
        mrs = mrs_values.values[0]

    bp_df.loc[bp_df["pNr"] == pnr, "mrs_1y"] = mrs

In [ ]:
bp_df.head()

In [ ]:
# for each pNr in bp_df, get durtation of monitoring by last_measure - first_measure
bp_df['timeBd']=pd.to_datetime(bp_df['timeBd'], format='%Y-%m-%d %H:%M:%S.%f')
monitoring_duration_df = bp_df.groupby('pNr')['timeBd'].agg(['min', 'max']).reset_index()
monitoring_duration_df['monitoring_duration'] = (monitoring_duration_df['max'] - monitoring_duration_df['min']).dt.total_seconds() / 60  # convert to minutes

In [ ]:
main_df = bp_df.merge(registry_df, 
                       left_on=['SOS-CENTER-YEAR-NO.', 'JoinedName', 'Date_birth'], 
                       right_on=['SOS-CENTER-YEAR-NO.', 'Name', 'Date_birth'], 
                       how='left')

#### Compute timings

In [ ]:
main_df['Date_DCI_ischemia_first_image'] = pd.to_datetime(main_df['Date_DCI_ischemia_first_image'], errors='coerce', format='%Y-%m-%d')
main_df['Time_DCI_ischemia_first_image'] = pd.to_datetime(main_df['Time_DCI_ischemia_first_image'], errors='coerce', format='%H:%M:%S')

main_df['Date_DCI_infarct_first_image'] = pd.to_datetime(main_df['Date_DCI_infarct_first_image'], errors='coerce', format='%Y-%m-%d')
main_df['Date_DCI_infarct_first_image'] = pd. to_datetime(main_df['Date_DCI_infarct_first_image'],errors='coerce', format='%H:%M:%S')

main_df['timestamp_ischemia'] = pd.to_datetime(
    main_df['Date_DCI_ischemia_first_image'].astype(str) + ' ' + main_df['Time_DCI_ischemia_first_image'].astype(str),
    errors='coerce'
)

main_df['timestamp_infarction'] =  pd.to_datetime(
    main_df['Date_DCI_infarct_first_image'].astype(str) + ' ' + main_df['Time_DCI_infarct_first_image'].astype(str),
    errors='coerce'
)

main_df['timeBd']=pd.to_datetime(main_df['timeBd'], format='%Y-%m-%d %H:%M:%S.%f')

main_df['timeBd'] = main_df['timeBd'].dt.tz_localize(None)
main_df['timestamp_ischemia'] = main_df['timestamp_ischemia'].dt.tz_localize(None)

main_df['time_difference_ischemia']=main_df['timestamp_ischemia'] - main_df['timeBd']
main_df['time_difference_ischemia']=main_df['time_difference_ischemia'].dt.total_seconds() / 60

main_df['time_difference_infarction']=main_df['timestamp_infarction']-main_df['timeBd']
main_df['time_difference_infarction']=main_df['time_difference_infarction'].dt.total_seconds() / 60

main_df = main_df.sort_values(by=['pNr', 'timeBd'], ascending=True)

main_df['T0'] = main_df.groupby('pNr')['timeBd'].transform('min')
main_df['relative_time'] = main_df['timeBd'] - main_df['T0']
main_df['relative_time'] = main_df['relative_time'].dt.total_seconds() / 60
main_df['relative_time'] = pd.to_numeric(main_df['relative_time'], errors='coerce')

main_df['first_Th_relative_date'] = (pd.to_datetime(main_df['Date_First_Th']) - main_df['T0']).dt.total_seconds() / 60

#### Compute pressure x time product

In [ ]:
main_df['delta_time'] = main_df['timeBd'].shift(-1) - main_df['timeBd']
main_df['delta_time'] = main_df['delta_time'].dt.total_seconds() / 60

In [ ]:
main_df['product'] = main_df['delta_time'] * main_df['systole']


In [ ]:
working_df = main_df[['relative_time', 'pNr', 'delta_time', 'systole', 'product', 'DCI_YN_verified', 'mrs_1y', 'first_Th_relative_date']]

# Systolic blood pressure

#### First 24h

In [ ]:
# analysis for first 24 hours of monitoring
working_df_in_first_24h_monitoring = working_df[working_df['relative_time'] <= 24 * 60]  # 24 hours in minutes

events_df_in_first_24h_monitoring = define_events_multiple_thresholds(working_df_in_first_24h_monitoring,
                                           intensity_thresholds=[140, 150, 160, 170, 180, 190, 200, 210, 220],
                                           parameter_name='systole')

duration_thresholded_events_df_in_first_24h_monitoring = multiple_duration_thresholds(events_df_in_first_24h_monitoring,
                                                                                          duration_thresholds=[1, 5, 10, 15, 20, 30, 60, 120, 180])

event_counts_in_first_24h_monitoring_df = count_events(duration_thresholded_events_df_in_first_24h_monitoring)

# correlation between event count and mRS_1y
event_count_correlation_first_24h_monitoring_df = event_count_to_mrs_correlation(event_counts_in_first_24h_monitoring_df)

#  correlation between event product and mRS_1y
event_product_correlation_first_24h_monitoring_df = event_product_to_mrs_correlation(duration_thresholded_events_df_in_first_24h_monitoring)

In [ ]:
# plot intensity threshold on x-axis, duration threshold on y-axis, and color as correlation coefficient (count to mRS_1y)
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(event_count_correlation_first_24h_monitoring_df.pivot_table(
    index='duration_threshold', 
    columns='intensity_threshold', 
    values='correlation_coefficient'
).reindex(index=sorted(event_count_correlation_first_24h_monitoring_df['duration_threshold'].unique(), reverse=True)),
    annot=True, cmap='seismic', center=0, ax=ax)

In [ ]:
relative_duration_in_correlated_events_df_before_24h = relative_duration_in_correlated_events(
    duration_thresholded_events_df_in_first_24h_monitoring,
    event_count_correlation_first_24h_monitoring_df,
    monitoring_duration_df,
    threshold=0
)

# Univariate association of duration with mRS_1y (ordinal regression)
temp_df = relative_duration_in_correlated_events_df_before_24h[['positively_correlated_event_proportion_of_monitoring_duration', 'negatively_correlated_event_proportion_of_monitoring_duration', 'mrs_1y']].dropna()

pos_event_duration_model = OrderedModel(
    temp_df['mrs_1y'],
    temp_df[['positively_correlated_event_proportion_of_monitoring_duration']],
    distr='logit'
)
neg_event_duration_model = OrderedModel(
    temp_df['mrs_1y'],
    temp_df[['negatively_correlated_event_proportion_of_monitoring_duration']],
    distr='logit'
)
pos_event_duration_result = pos_event_duration_model.fit(method='bfgs')
neg_event_duration_result = neg_event_duration_model.fit(method='bfgs')
print("Positive Event Duration Model Summary:")
print(pos_event_duration_result.summary())
print("Negative Event Duration Model Summary:")
print(neg_event_duration_result.summary())


In [ ]:
# mutlivariable model with Age, WFNS, Fisher_Score, Coiling, Clipping
temp_df = relative_duration_in_correlated_events_df_before_24h.merge(
    main_df[['pNr', 'Age', 'WFNS', 'Fisher_Score', 'Coiling', 'Clipping']],
    on='pNr',
    how='left'
)
temp_df = temp_df[['positively_correlated_event_proportion_of_monitoring_duration',
                   'negatively_correlated_event_proportion_of_monitoring_duration',
                   'mrs_1y', 'Age', 'WFNS', 'Fisher_Score', 'Coiling', 'Clipping']].dropna()

temp_df['Age'] = pd.to_numeric(temp_df['Age'], errors='coerce')
temp_df['WFNS'] = pd.to_numeric(temp_df['WFNS'], errors='coerce')
temp_df['Fisher_Score'] = pd.to_numeric(temp_df['Fisher_Score'], errors='coerce')
temp_df['Coiling'] = pd.to_numeric(temp_df['Coiling'], errors='coerce')
temp_df['Clipping'] = pd.to_numeric(temp_df['Clipping'], errors='coerce')

pos_event_duration_model_multivariable = OrderedModel(
    temp_df['mrs_1y'],
    temp_df[['positively_correlated_event_proportion_of_monitoring_duration',
             'Age', 'WFNS', 'Fisher_Score', 'Coiling', 'Clipping']],
    distr='logit'
)
neg_event_duration_model_multivariable = OrderedModel(
    temp_df['mrs_1y'],
    temp_df[['negatively_correlated_event_proportion_of_monitoring_duration',
             'Age', 'WFNS', 'Fisher_Score', 'Coiling', 'Clipping']],
    distr='logit'
)
pos_event_duration_result_multivariable = pos_event_duration_model_multivariable.fit(method='bfgs')
neg_event_duration_result_multivariable = neg_event_duration_model_multivariable.fit(method='bfgs')
print("Positive Event Duration Multivariable Model Summary:")
print(pos_event_duration_result_multivariable.summary())
print("Negative Event Duration Multivariable Model Summary:")
print(neg_event_duration_result_multivariable.summary())

#### After 24h

In [ ]:
# analysis for 24h-to-end of monitoring

working_df_after_24h_monitoring = working_df[working_df['relative_time'] > 24 * 60]  # 24 hours in minutes
events_df_after_24h_monitoring = define_events_multiple_thresholds(working_df_after_24h_monitoring,
                                           intensity_thresholds=[140, 150, 160, 170, 180, 190, 200, 210, 220],
                                            #  intensity_thresholds=range(140, 221, 1),
                                           parameter_name='systole')
duration_thresholded_events_df_after_24h_monitoring = multiple_duration_thresholds(events_df_after_24h_monitoring,
                                                                                          duration_thresholds=[1, 5, 10, 15, 20, 30, 60, 120, 180])

event_counts_df = count_events(duration_thresholded_events_df_after_24h_monitoring)

events_total_duration_df = total_event_duration(duration_thresholded_events_df_after_24h_monitoring)

# correlation between event count and mRS_1y
event_count_correlation_after_24h_monitoring_df = event_count_to_mrs_correlation(event_counts_df)

# correlation between event product and mRS_1y
event_product_correlation_after_24h_monitoring_df = event_product_to_mrs_correlation(duration_thresholded_events_df_after_24h_monitoring)

# correlation between relative total event duration and mRS_1y
events_total_duration_df = total_event_duration(duration_thresholded_events_df_after_24h_monitoring)
events_total_duration_df = events_total_duration_df.merge(
    monitoring_duration_df[['pNr', 'monitoring_duration']],
    on='pNr',
    how='left'
)
events_total_duration_df['total_event_duration_proportion'] = events_total_duration_df['total_event_duration'] / events_total_duration_df['monitoring_duration']

event_relative_duration_to_mrs_correlation_df = event_relative_duration_to_mrs_correlation(events_total_duration_df)


In [ ]:
# plot intensity threshold on x-axis, duration threshold on y-axis, and color as correlation coefficient (count to mRS_1y)

sns.heatmap(event_count_correlation_after_24h_monitoring_df.pivot_table(
    index='duration_threshold', 
    columns='intensity_threshold', 
    values='correlation_coefficient'
).reindex(index=sorted(event_count_correlation_after_24h_monitoring_df['duration_threshold'].unique(), reverse=True)),
    annot=True, cmap='seismic', center=0)

Association of red zone with outcome

In [ ]:
relative_duration_in_correlated_events_df_after_24h_monitoring = relative_duration_in_correlated_events(duration_thresholded_events_df_after_24h_monitoring,
                                                                                           event_count_correlation_after_24h_monitoring_df,
                                                                                           monitoring_duration_df, threshold=0)
# event_counts_with_correlation_after_24h_monitoring = total_correlated_event_counts(event_counts_df, event_count_correlation_after_24h_monitoring_df)


In [ ]:
# Univariate association of duration with mRS_1y (ordinal regression)
temp_df = relative_duration_in_correlated_events_df_after_24h_monitoring[['positively_correlated_event_proportion_of_monitoring_duration', 'negatively_correlated_event_proportion_of_monitoring_duration', 'mrs_1y']].dropna()

pos_event_duration_model = OrderedModel(
    temp_df['mrs_1y'],
    temp_df[['positively_correlated_event_proportion_of_monitoring_duration']],
    distr='logit'
)
neg_event_duration_model = OrderedModel(
    temp_df['mrs_1y'],
    temp_df[['negatively_correlated_event_proportion_of_monitoring_duration']],
    distr='logit'
)
pos_event_duration_result = pos_event_duration_model.fit(method='bfgs')
neg_event_duration_result = neg_event_duration_model.fit(method='bfgs')
print("Positive Event Duration Model Summary:")
print(pos_event_duration_result.summary())
print("Negative Event Duration Model Summary:")
print(neg_event_duration_result.summary())

In [ ]:
# mutlivariable model with Age, WFNS, Fisher_Score, Coiling, Clipping
temp_df = relative_duration_in_correlated_events_df_after_24h_monitoring.merge(
    main_df[['pNr', 'Age', 'WFNS', 'Fisher_Score', 'Coiling', 'Clipping']],
    on='pNr',
    how='left'
)
temp_df = temp_df[['positively_correlated_event_proportion_of_monitoring_duration',
                                                              'negatively_correlated_event_proportion_of_monitoring_duration',
                                                              'mrs_1y', 'Age', 'WFNS', 'Fisher_Score', 'Coiling', 'Clipping']].dropna()
temp_df['Age'] = pd.to_numeric(temp_df['Age'], errors='coerce')
temp_df['WFNS'] = pd.to_numeric(temp_df['WFNS'], errors='coerce')
temp_df['Fisher_Score'] = pd.to_numeric(temp_df['Fisher_Score'], errors='coerce')
temp_df['Coiling'] = pd.to_numeric(temp_df['Coiling'], errors='coerce')
temp_df['Clipping'] = pd.to_numeric(temp_df['Clipping'], errors='coerce')


pos_event_duration_model_multivar = OrderedModel(
    temp_df['mrs_1y'],
    temp_df[['positively_correlated_event_proportion_of_monitoring_duration', 'Age', 'WFNS', 'Fisher_Score', 'Coiling', 'Clipping']],
    distr='logit'
)
neg_event_duration_model_multivar = OrderedModel(
    temp_df['mrs_1y'],
    temp_df[['negatively_correlated_event_proportion_of_monitoring_duration', 'Age', 'WFNS', 'Fisher_Score', 'Coiling', 'Clipping']],
    distr='logit'
)
pos_event_duration_result_multivar = pos_event_duration_model_multivar.fit(method='bfgs')
neg_event_duration_result_multivar = neg_event_duration_model_multivar.fit(method='bfgs')
print("Positive Event Duration Multivariable Model Summary:")
print(pos_event_duration_result_multivar.summary())
print("Negative Event Duration Multivariable Model Summary:")
print(neg_event_duration_result_multivar.summary())

#### Before aneurysm securisation 

(as exact time of securisation is not known, add 24h to date)

In [ ]:
ax = (working_df['first_Th_relative_date']/60).hist(bins=100)

In [ ]:
working_df_before_aneurym_secured = working_df[working_df['relative_time'] < (working_df['first_Th_relative_date'] + 24 * 60)]  # 24 hours in minutes

events_df_before_aneurym_secured = define_events_multiple_thresholds(working_df_before_aneurym_secured,
                                           intensity_thresholds=[140, 150, 160, 170, 180, 190, 200, 210, 220],
                                           parameter_name='systole')
duration_thresholded_events_df_before_aneurym_secured = multiple_duration_thresholds(events_df_before_aneurym_secured,
                                                                                          duration_thresholds=[1, 5, 10, 15, 20, 30, 60, 120, 180])
event_counts_before_aneurym_secured_df = count_events(duration_thresholded_events_df_before_aneurym_secured)
# correlation between event count and mRS_1y
event_count_correlation_before_aneurym_secured_df = event_count_to_mrs_correlation(event_counts_before_aneurym_secured_df)

In [ ]:
# plot intensity threshold on x-axis, duration threshold on y-axis, and color as correlation coefficient (count to mRS_1y)
sns.heatmap(event_count_correlation_before_aneurym_secured_df.pivot_table(
    index='duration_threshold', 
    columns='intensity_threshold', 
    values='correlation_coefficient'
).reindex(index=sorted(event_count_correlation_before_aneurym_secured_df['duration_threshold'].unique(), reverse=True)),
    annot=True, cmap='seismic', center=0)

In [ ]:
pos_event_duration_result, neg_event_duration_result, pos_event_duration_result_multivariable, neg_event_duration_result_multivariable = decision_boundary_analysis(duration_thresholded_events_df_before_aneurym_secured, event_count_correlation_before_aneurym_secured_df, monitoring_duration_df, main_df, verbose=True)

#### After aneurysm secured

In [ ]:
# after aneurysm secured
working_df_after_aneurym_secured = working_df[working_df['relative_time'] >= (working_df['first_Th_relative_date'] + 24 * 60)]  # 24 hours in minutes
events_df_after_aneurym_secured = define_events_multiple_thresholds(working_df_after_aneurym_secured,
                                           intensity_thresholds=[140, 150, 160, 170, 180, 190, 200, 210, 220],
                                           parameter_name='systole')
duration_thresholded_events_df_after_aneurym_secured = multiple_duration_thresholds(events_df_after_aneurym_secured,
                                                                                          duration_thresholds=[1, 5, 10, 15, 20, 30, 60, 120, 180])
event_counts_after_aneurym_secured_df = count_events(duration_thresholded_events_df_after_aneurym_secured)
# correlation between event count and mRS_1y
event_count_correlation_after_aneurym_secured_df = event_count_to_mrs_correlation(event_counts_after_aneurym_secured_df)

In [ ]:
# plot intensity threshold on x-axis, duration threshold on y-axis, and color as correlation coefficient (count to mRS_1y)
sns.heatmap(event_count_correlation_after_aneurym_secured_df.pivot_table(
    index='duration_threshold', 
    columns='intensity_threshold', 
    values='correlation_coefficient'
).reindex(index=sorted(event_count_correlation_after_aneurym_secured_df['duration_threshold'].unique(), reverse=True)),
    annot=True, cmap='seismic', center=0)

In [ ]:
pos_event_duration_result, neg_event_duration_result, pos_event_duration_result_multivariable, neg_event_duration_result_multivariable = decision_boundary_analysis(duration_thresholded_events_df_after_aneurym_secured, event_count_correlation_after_aneurym_secured_df, monitoring_duration_df, main_df, verbose=True)